Instead of:

`face A vs face B`

We will do:

`Unknown face → compare against database → find best match`

This is how real systems work.

### Step 1 — Create a Small Face Database

We will:

- Extract embeddings
- Store them
- Associate them with names

### Step 2 — Build Database Embeddings

In [12]:
import torch
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device)

resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [14]:
import os

database = {}

for person in os.listdir('faces/single'):
    for file in os.listdir(f'faces/single/{person}'):
        img = Image.open(os.path.join(f'faces/single/{person}', file))
        face = mtcnn(img)
        
        if face is None:
            continue
        
        face = face.unsqueeze(0).to(device)
        
        with torch.no_grad():
            embedding = resnet(face)
            
        database[file] = embedding
        
print(list(database.keys()))

['abhishek_bachchan1.png', 'abhishek_bachchan2.png', 'abhishek_bachchan3.png', 'ab1.png', 'ab2.png', 'ab3.png', 'ab4.png', 'ab5.png', 'jaya_bachchan1.png', 'jaya_bachchan2.png', 'jaya_bachchan3.png', 'jaya_bachchan4.png', 'shail1.jpg', 'shail2.jpg']


We:

- Read each image
- Extracted face
- Converted it to embedding
- Stored embedding in dictionary

So now database looks like:
<pre>
{
  "shail1": tensor([...]),
  "friend1": tensor([...])
}
</pre>
Each person = stored 512-dim vector.

### Step 3 — Recognition Function

In [17]:
def recognize_face(query_image, database, threshold=0.8):
    
    img = Image.open(query_image)
    face = mtcnn(img)
    
    if face is None:
        return "No Face Detected", None
    
    face = face.unsqueeze(0).to(device)
    
    with torch.no_grad():
        query_embedding = resnet(face)
        
    best_match = None
    min_distance = float('inf')
    
    for name, db_embedding in database.items():
        
        if(name == query_image.split("/")[-1]):
            continue
        
        distance = torch.norm(query_embedding - db_embedding).item()
        
        if distance < min_distance:
            min_distance = distance
            best_match = name
        
    if min_distance < threshold:
        return best_match, min_distance
    else:
        return "Unknown", min_distance
    

In [18]:
name, distance = recognize_face("faces/single/amitabh_bachchan/ab1.png", database)

print("Name: ", name)
print("Distance: ", distance)

Name:  ab2.png
Distance:  0.3925554156303406
